可以同时输出不同时间的单日期日报。做日报使用。

CPI维度为转化回传 两个地址

In [3]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
import time
import os

# 读取文件-------------------------------------------------------------------------------------------------------------------------------------
country_df = pd.read_excel(r"C:\Users\lili.li\Downloads\区域-国家.xlsx")
data_df = pd.read_excel(r"C:\Users\lili.li\Downloads\国家_地区报表_2025-07-04-2025-07-06_分日 (5).xlsx")#请求
conversion_df = pd.read_excel(r"C:\Users\lili.li\Downloads\国家_地区报表_2025-07-04-2025-07-06_分日 (6).xlsx")  # 转化回传文件

# 合并国家/地区数据
merged_df = data_df.merge(
    country_df[['国家/地区(CN)', '所属区域（运营）']],
    left_on='国家/地区',
    right_on='国家/地区(CN)',
    how='left'
).drop(columns=['国家/地区(CN)']).rename(columns={'所属区域（运营）': '区域'})

# 合并转化回传数据的花费和激活量
merged_df = merged_df.merge(
    conversion_df[['时间', '国家/地区', '产品名称', '花费', '激活量']],
    on=['时间', '国家/地区', '产品名称'],
    how='left',
    suffixes=('', '（转化）')
).rename(columns={'花费（转化）': '花费（转化）', '激活量（转化）': '激活量（转化）'})

# 过滤区域————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
target_regions = ['俄罗斯', '东南亚', '亚太', '拉美']
filtered_df = merged_df[merged_df['区域'].isin(target_regions)]
#filtered_df = merged_df

# 按日期、区域、产品名称汇总
result = pd.DataFrame()
dates = filtered_df['时间'].unique()
for date in sorted(dates):
    date_df = filtered_df[filtered_df['时间'] == date]
    date_result = date_df.groupby(['时间', '区域', '产品名称']).agg({
        '花费': 'sum',
        '激活量': 'sum',
        '收益': 'sum',
        '花费（转化）': 'sum',
        '激活量（转化）': 'sum'
    }).reset_index()
    region_totals = date_df.groupby(['时间', '区域']).agg({
        '花费': 'sum',
        '激活量': 'sum',
        '收益': 'sum',
        '花费（转化）': 'sum',
        '激活量（转化）': 'sum'
    }).reset_index()
    region_totals['产品名称'] = ''
    date_combined = pd.concat([date_result, region_totals], ignore_index=True).sort_values(by=['区域', '产品名称'], na_position='last')
    date_combined.loc[date_combined['产品名称'] != '', '区域'] = ''
    result = pd.concat([result, date_combined], ignore_index=True)
    empty_row = pd.DataFrame([[None] * len(result.columns)], columns=result.columns)
    result = pd.concat([result, empty_row], ignore_index=True)

# 计算 CPI（基于转化回传数据）
result['CPI（回传）'] = result.apply(
    lambda x: round(x['花费（转化）'] / x['激活量（转化）'], 2) if pd.notna(x['激活量（转化）']) and x['激活量（转化）'] != 0 else '#DIV/0!',
    axis=1
)

# 计算 ROAS
result['请求ROAS'] = result.apply(
    lambda x: f"{x['收益'] / x['花费'] * 100:.2f}%" if pd.notna(x['花费']) and x['花费'] != 0 else '#DIV/0!',
    axis=1
)
result = result.rename(columns={'收益': '收益（广告请求）'})
# 删除花费（转化）和激活量（转化）列
result = result.drop(columns=['花费（转化）', '激活量（转化）'])

# 添加总计行
total = filtered_df.groupby(['时间']).agg({
    '花费': 'sum',
    '激活量': 'sum',
    '收益': 'sum',
    '花费（转化）': 'sum',
    '激活量（转化）': 'sum'
}).reset_index()
total['区域'] = '总计'
total['产品名称'] = ''
total['CPI（回传）'] = total.apply(
    lambda x: round(x['花费（转化）'] / x['激活量（转化）'], 2) if x['激活量（转化）'] != 0 else '#DIV/0!',
    axis=1
)
total['请求ROAS'] = total.apply(
    lambda x: f"{x['收益'] / x['花费'] * 100:.2f}%" if x['花费'] != 0 else '#DIV/0!',
    axis=1
)
total = total.rename(columns={'收益': '收益（广告请求）'})
total = total.drop(columns=['花费（转化）', '激活量（转化）'])
final_result = pd.DataFrame()
for date in sorted(dates):
    date_data = result[result['时间'] == date]
    date_total = total[total['时间'] == date]
    final_result = pd.concat([final_result, date_data, date_total], ignore_index=True)


# 保存路径————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————————
output_file = r"C:\Users\lili.li\Desktop\日报\output-多比特-0704-0706.xlsx"
if os.path.exists(output_file):
    print(f"检测到文件 {output_file} 已存在，请关闭文件或删除后重试。")
else:
    try:
        with pd.ExcelWriter(output_file) as writer:
            merged_df.to_excel(writer, sheet_name='原始数据', index=False)
            final_result.to_excel(writer, sheet_name='数据展示', index=False)
        print(f"Excel 文件已保存到: {os.path.abspath(output_file)}")
        time.sleep(1)

        # 美化 Excel
        wb = load_workbook(output_file)
        if '数据展示' in wb.sheetnames:
            ws = wb['数据展示']
            header_fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
            region_fill = PatternFill(start_color="E6F3FA", end_color="E6F3FA", fill_type="solid")
            total_fill = PatternFill(start_color="FFFACD", end_color="FFFACD", fill_type="solid")
            bold_font = Font(bold=True)
            center_align = Alignment(horizontal='center', vertical='center')

            new_ws = wb.create_sheet('数据展示新', 0)
            headers = list(final_result.columns)
            for date in sorted(dates):
                new_ws.append(headers)
                date_data = final_result[final_result['时间'] == date]
                for _, row in date_data.iterrows():
                    new_ws.append(list(row))
                new_ws.append([None] * len(headers))

            header_rows = []
            for row in range(1, new_ws.max_row + 1):
                cell_value = new_ws.cell(row=row, column=1).value
                if cell_value == '时间':
                    header_rows.append(row)

            for row in new_ws.iter_rows(min_row=1, max_row=new_ws.max_row):
                row_num = row[0].row
                region_cell = new_ws.cell(row=row_num, column=2).value
                if row_num in header_rows:
                    for cell in row:
                        cell.fill = header_fill
                        cell.font = bold_font
                        cell.alignment = center_align
                elif region_cell in ['亚太', '俄罗斯', '拉美', '东南亚','中东非','东北欧','西欧',]:
                    for cell in row:
                        cell.fill = region_fill
                        cell.alignment = center_align
                elif region_cell == '总计':
                    for cell in row:
                        cell.fill = total_fill
                        cell.font = bold_font
                        cell.alignment = center_align
                elif pd.isna(region_cell):
                    continue

            for col in new_ws.columns:
                max_length = 0
                column = col[0].column_letter
                for cell in col:
                    try:
                        if len(str(cell.value)) > max_length:
                            max_length = len(str(cell.value))
                    except:
                        pass
                adjusted_width = (max_length + 2) * 1.2
                new_ws.column_dimensions[column].width = adjusted_width

            wb.remove(ws)
            new_ws.title = '数据展示'
            wb.save(output_file)
            print("美化后的结果已保存")
        else:
            print("错误：未找到 '数据展示' 工作表")
    except PermissionError as pe:
        print(f"权限错误: {pe}. 请确保文件未被占用，或以管理员权限运行。")
    except Exception as e:
        print(f"发生错误: {e}")

C:\Users\lili.li\.conda\envs\taidong\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\lili.li\.conda\envs\taidong\lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
C:\Users\lili.li\AppData\Local\Temp\ipykernel_42836\1412483021.py:57: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result = pd.concat([result, empty_row], ignore_index=True)
C:\Users\lili.li\AppData\Local\Temp\ipykernel_42836\1412483021.py:57: FutureWarning: The behavior of Data

Excel 文件已保存到: C:\Users\lili.li\Desktop\日报\output-多比特-0704-0706.xlsx
美化后的结果已保存
